# WaRS — huấn luyện trên Kaggle

**Settings:** Accelerator = GPU (T4 hoặc P100), Internet = On.
**Add Input:** dataset MaSTr1325 (ảnh + mask) và dataset USV-VN (thư mục `usv_vn/` có `train/valid/test`), cả hai upload dạng *private dataset*.
Repo private → tạo Kaggle Secret tên `GITHUB_TOKEN` (Add-ons → Secrets).

Kaggle không giữ `/kaggle/working` giữa các phiên: bấm **Save Version (Save & Run All)** để lưu output, rồi ở phiên sau *Add Input* chính output đó để chạy tiếp từ `last.pt` (ô số 3).

In [ ]:
GITHUB_USER = "<github-username>"   # sửa
REPO = "WaRS"
import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"
except Exception:
    url = f"https://github.com/{GITHUB_USER}/{REPO}.git"   # repo public
subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/kaggle/working/WaRS"], check=True)
%cd /kaggle/working/WaRS
!pip install -q albumentations onnx onnxruntime onnxsim
!mkdir -p weights && curl -sL -o weights/ewasr_resnet18.pth https://github.com/tersekmatija/eWaSR/releases/download/0.1.0/ewasr_resnet18.pth

In [ ]:
# Tự tìm thư mục MaSTr1325 (chứa *m.png) và USV-VN trong /kaggle/input
from pathlib import Path
mask_dir = next(p.parent for p in Path("/kaggle/input").rglob("*m.png"))
image_dir = next(p.parent for p in Path("/kaggle/input").rglob("0001.jpg") if p.parent != mask_dir)
mastr_root = Path(os.path.commonpath([mask_dir, image_dir]))
usv_root = next((p.parent.parent for p in Path("/kaggle/input").rglob("left_*.jpg")), None).parent
print("MaSTr:", mastr_root, image_dir.relative_to(mastr_root), mask_dir.relative_to(mastr_root))
print("USV-VN:", usv_root)

In [ ]:
# (Phiên sau) chạy tiếp: chép last.pt từ output phiên trước đã Add Input
RUN = "ewasr_h"
OUT = f"/kaggle/working/runs/{RUN}"
prev = list(Path("/kaggle/input").rglob(f"runs/{RUN}/last.pt"))
if prev:
    os.makedirs(OUT, exist_ok=True)
    !cp -n {prev[0].parent}/* {OUT}/
    print("Tiếp tục từ", prev[0])

In [ ]:
!python tools/train.py --config configs/{RUN}.yaml --out {OUT} --time-limit-h 11 \
    --set data.mastr_root={mastr_root} data.image_dir={image_dir.relative_to(mastr_root)} \
          data.mask_dir={mask_dir.relative_to(mastr_root)} train.workers=4

In [ ]:
# Xuất ONNX + đánh giá nhanh trên USV-VN (zero-shot, chỉ Recall nếu chưa có water_edges.json)
!python tools/export_onnx.py --model {RUN} --weights {OUT}/best.pth --size 384 512 --out {OUT}/{RUN}_384x512.onnx
!python tools/predict.py --model {RUN} --weights {OUT}/best.pth --usv-root {usv_root} --split test --out {OUT}/pred_usv_test
!python tools/evaluate_usv.py --pred-masks {OUT}/pred_usv_test/masks --usv-root {usv_root} --split test --size-edges 56 70

## YOLOv8n trên USV-VN (mô hình đối chứng)

In [ ]:
!pip install -q ultralytics
import yaml
cfg = yaml.safe_load(open("configs/usv_vn_yolo.yaml")); cfg["path"] = str(usv_root)
yaml.safe_dump(cfg, open("/kaggle/working/usv_vn_yolo.yaml", "w"))
!yolo detect train data=/kaggle/working/usv_vn_yolo.yaml model=yolov8n.pt imgsz=640 epochs=100 project=/kaggle/working/runs/yolo name=usv_vn exist_ok=True
!python tools/yolo_predict.py --weights /kaggle/working/runs/yolo/usv_vn/weights/best.pt --usv-root {usv_root} --split test --out /kaggle/working/runs/yolo/pred_usv_test
!python tools/evaluate_usv.py --pred-boxes /kaggle/working/runs/yolo/pred_usv_test/boxes.json --usv-root {usv_root} --split test --size-edges 56 70
# ONNX cho Hailo Model Zoo (hailomz compile yolov8n --ckpt ... --hw-arch hailo8l --classes 1)
!yolo export model=/kaggle/working/runs/yolo/usv_vn/weights/best.pt format=onnx opset=11 imgsz=640